# IOAI — 2024 Final Stage Ciphers (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ciphered_lines.txt'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-final-stage-ciphers/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 암호 해독 — 모범답안 (Szyfry, trigram 언덕오르기)

동형사상 치환 암호(이모지→글자, 다대일). **글자 trigram 언어모델**(clear 코퍼스)로 매핑을 최적화한다:
빈도로 초기화 → 각 이모지에 대해, 그 이모지가 든 암호문 trigram 들의 로그가능도를 최대화하는 글자로 재배정하는
**언덕오르기**를 수렴까지 반복. 검증셋 글자정확도 ≈ **0.98**.

## 데이터 로드

In [ ]:
import numpy as np
from collections import Counter
corpus_clear    = [l.strip().lower()  for l in open("data/clear_lines.txt", encoding="utf-8")]
corpus_ciphered = [l.strip().lower()  for l in open("data/ciphered_lines.txt", encoding="utf-8")]
print("clear", len(corpus_clear), "| ciphered", len(corpus_ciphered))

## decipher_corpus — trigram 언덕오르기

In [ ]:
def decipher_corpus(clear_corpus, ciphered_corpus):
    chars = sorted(set("".join(clear_corpus))); ci = {c:i for i,c in enumerate(chars)}; C = len(chars)
    # 글자 trigram LM (clear 코퍼스)
    tf = np.zeros(C*C*C)
    for line in clear_corpus:
        if len(line) < 3: continue
        idx = np.fromiter((ci[c] for c in line), np.int64, len(line))
        np.add.at(tf, idx[:-2]*C*C + idx[1:-1]*C + idx[2:], 1.0)
    logP3 = np.log((tf.reshape(C,C,C)+0.1) / (tf.reshape(C,C,C)+0.1).sum(2, keepdims=True)).astype(np.float32)
    # 암호문 trigram 카운트 + 이모지별 위치 인덱스
    syms = sorted(set("".join(ciphered_corpus))); si = {s:i for i,s in enumerate(syms)}; S = len(syms)
    from collections import Counter as Ct
    c3 = Ct()
    for line in ciphered_corpus:
        idx = [si[s] for s in line]
        for a,b,c in zip(idx, idx[1:], idx[2:]): c3[(a,b,c)] += 1
    be = {e:[[[],[],[]],[[],[],[]],[[],[],[]]] for e in range(S)}
    for (a,b,c),n in c3.items():
        be[a][0][0].append(b); be[a][0][1].append(c); be[a][0][2].append(n)
        be[b][1][0].append(a); be[b][1][1].append(c); be[b][1][2].append(n)
        be[c][2][0].append(a); be[c][2][1].append(b); be[c][2][2].append(n)
    for e in range(S):
        for p in range(3):
            be[e][p] = [np.array(be[e][p][0]), np.array(be[e][p][1]), np.array(be[e][p][2], np.float32)]
    euni = np.array([Counter("".join(ciphered_corpus))[s] for s in syms], float)
    cuni = np.array([Counter("".join(clear_corpus))[c] for c in chars], float)
    # 빈도 초기화
    m = np.zeros(S, int); eo = np.argsort(-euni); co = np.argsort(-cuni)
    ecum = np.cumsum(euni[eo])/euni.sum(); ccum = np.cumsum(cuni[co])/cuni.sum(); j = 0
    for k,e in enumerate(eo):
        while j < C-1 and ecum[k] > ccum[j]: j += 1
        m[e] = co[j]
    def score(e):
        t = np.zeros(C, np.float32); o = be[e]
        b,cc,n = o[0]
        if len(n): t += logP3[:, m[b], m[cc]] @ n
        a,cc,n = o[1]
        if len(n): t += n @ logP3[m[a], :, m[cc]]
        a,b,n = o[2]
        if len(n): t += n @ logP3[m[a], m[b], :]
        return t
    for _ in range(15):
        changed = 0
        for e in np.argsort(-euni):
            bt = int(np.argmax(score(e)))
            if bt != m[e]: m[e] = bt; changed += 1
        if changed == 0: break
    lut = {syms[e]: chars[m[e]] for e in range(S)}
    return ["".join(lut.get(s, s) for s in line) for line in ciphered_corpus]

## 해독 → submission.txt

In [ ]:
deciphered = decipher_corpus(corpus_clear, corpus_ciphered)
with open("submission.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(deciphered))
print("saved submission.txt", len(deciphered), "lines")

4-gram·재시작(restart)·EM 을 더하면 더 오를 수 있다. 원본은 4개 독립 암호에서 char-accuracy 로 채점.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.txt']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)